In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_DIR = Path("/content/candidates.db")
RESUMES_DIR = PROJECT_DIR / "resumes"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RESUMES_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = PROJECT_DIR / "candidates.db"

from google.colab import userdata
userdata.get('OPENROUTER_API_KEY')

Mounted at /content/drive


'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'

In [2]:
!pip install -q "mcp[cli]<2" google-genai python-dotenv
!apt-get -qq update && apt-get -qq install -y nodejs npm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package node-fast-deep-equal.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../000-node-fast-deep-equal_3.1.3-3_all.deb ...
Unpacking node-fast-deep-equal (3.1.3-3) ...
Selecting previously unselected package node-json-schema-traverse.
Preparing to unpack .../001-node-json-schema-traverse_1.0.0-3_all.deb ...
Unpacking node-json-schema-traverse (1.0.0-3) ...
Selecting previously unselected package node-jsonify.
Preparing to unpack .../002-node-jsonify_0.0.1-1_all.deb ...
Unpacking node-jsonify (0.0.1-1) ...
Selecting previously unselected package node-json-stable-stringify.
Preparing to unpack .../003-node-json-stable-stringify_1.0.2+repack1+~cs1.0.34-2_all.deb ...
Unpacking node-js

In [3]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [7]:
import sqlite3
from pathlib import Path
from typing import Literal
from mcp.server.fastmcp import FastMCP
import nest_asyncio
import sys

# Patch the asyncio loop to allow nested calls within Google Colab
nest_asyncio.apply()

mcp = FastMCP("candidate-tracker")

# Use the DB_PATH from the first cell or fallback to a local one
try:
    # DB_PATH is defined in the first cell as PROJECT_DIR / "candidates.db"
    _ = DB_PATH
except NameError:
    DB_PATH = Path("candidates.db")


def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


@mcp.resource("hiring://pipeline-summary")
def pipeline_summary() -> str:
    """Read-only summary of candidate counts and average scores grouped by hiring stage."""
    with get_conn() as conn:
        rows = conn.execute(
            """
            SELECT stage, COUNT(*) AS total, ROUND(AVG(score), 1) AS avg_score
            FROM candidates
            GROUP BY stage
            """
        ).fetchall()
    if not rows:
        return "No candidates in pipeline."
    return "\n".join(
        f"{r['stage']}: {r['total']} candidates (avg score: {r['avg_score']})"
        for r in rows
    )


@mcp.tool()
def list_candidates(
    stage: Literal["All", "Applied", "Shortlisted", "Interview", "Offer", "Rejected"] = "All",
    role_keyword: str = "",
    min_score: int = 0,
) -> list[dict]:
    """Fetch candidates from SQLite, optionally filtered by stage, role keyword, or minimum score (0-100)."""
    query = "SELECT * FROM candidates WHERE COALESCE(score, 0) >= ?"
    params: list = [min_score]

    if stage != "All":
        query += " AND stage = ?"
        params.append(stage)
    if role_keyword:
        query += " AND role LIKE ?"
        params.append(f"%{role_keyword}%")

    query += " ORDER BY score DESC"
    with get_conn() as conn:
        rows = conn.execute(query, params).fetchall()
    return [dict(r) for r in rows]


@mcp.tool()
def add_candidate(
    name: str,
    email: str,
    role: str,
    experience_years: float,
    skills: str,
    stage: Literal["Applied", "Shortlisted", "Interview", "Offer", "Rejected"] = "Applied",
    score: int = 50,
    expected_ctc_lpa: float | None = None,
    notes: str = "",
    applied_date: str = "2026-11-26",
) -> dict:
    """Insert a new candidate record into SQLite with a score between 0 and 100."""
    if not (0 <= score <= 100):
        return {"status": "error", "message": "score must be between 0 and 100"}

    with get_conn() as conn:
        cursor = conn.execute(
            """
            INSERT INTO candidates (
                name, email, role, experience_years, skills,
                stage, score, expected_ctc_lpa, notes, applied_date
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                name, email, role, experience_years, skills,
                stage, score, expected_ctc_lpa, notes, applied_date,
            ),
        )
        new_id = cursor.lastrowid
    return {"status": "created", "candidate_id": new_id, "name": name, "stage": stage, "score": score}


@mcp.tool()
def update_candidate_stage(
    candidate_id: int,
    new_stage: Literal["Applied", "Shortlisted", "Interview", "Offer", "Rejected"],
    score: int | None = None,
) -> dict:
    """Update a candidate's hiring stage (and optionally their 0-100 score) by ID."""
    if score is not None and not (0 <= score <= 100):
        return {"status": "error", "message": "score must be between 0 and 100"}

    with get_conn() as conn:
        if score is not None:
            cursor = conn.execute(
                "UPDATE candidates SET stage = ?, score = ? WHERE id = ?",
                (new_stage, score, candidate_id),
            )
        else:
            cursor = conn.execute(
                "UPDATE candidates SET stage = ? WHERE id = ?",
                (new_stage, candidate_id),
            )
        if cursor.rowcount == 0:
            return {"status": "error", "message": f"Candidate ID {candidate_id} not found"}
    return {"status": "updated", "candidate_id": candidate_id, "new_stage": new_stage, "score": score}


# Only run using stdio server model if launched as a standalone file, not when evaluating cells inside Colab
if __name__ == "__main__" and not hasattr(sys, "ps1"):
    mcp.run(transport="stdio")
else:
    print("MCP candidate-tracker loaded successfully inside Colab notebook (interactive mode).")

MCP candidate-tracker loaded successfully inside Colab notebook (interactive mode).


In [14]:
import json
import os
import sys
import traceback
from contextlib import AsyncExitStack
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI

# Ensure paths are correctly created on the local workspace
REAL_PROJECT_DIR = Path("/content/mcp_candidate_tracker")
REAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RESUMES_DIR = REAL_PROJECT_DIR / "resumes"
RESUMES_DIR.mkdir(parents=True, exist_ok=True)

# Write the actual server.py dynamically to the disk so it can be executed as a background subprocess
SERVER_CODE = """
import sqlite3
from pathlib import Path
from typing import Literal
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("candidate-tracker")
DB_PATH = Path("/content/mcp_candidate_tracker/candidates.db")

def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# Initialize database schema if not exists
with get_conn() as conn:
    conn.execute(
        'CREATE TABLE IF NOT EXISTS candidates ('
        'id INTEGER PRIMARY KEY AUTOINCREMENT, '
        'name TEXT NOT NULL, '
        'email TEXT NOT NULL, '
        'role TEXT NOT NULL, '
        'experience_years REAL, '
        'skills TEXT, '
        'stage TEXT DEFAULT "Applied", '
        'score INTEGER DEFAULT 50, '
        'expected_ctc_lpa REAL, '
        'notes TEXT, '
        'applied_date TEXT'
        ')'
    )

@mcp.resource("hiring://pipeline-summary")
def pipeline_summary() -> str:
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT stage, COUNT(*) AS total, ROUND(AVG(score), 1) AS avg_score FROM candidates GROUP BY stage"
        ).fetchall()
    if not rows:
        return "No candidates in pipeline."
    return "\\n".join(f"{r['stage']}: {r['total']} candidates (avg score: {r['avg_score']})" for r in rows)

@mcp.tool()
def list_candidates(stage: str = "All", role_keyword: str = "", min_score: int = 0) -> list[dict]:
    query = "SELECT * FROM candidates WHERE COALESCE(score, 0) >= ?"
    params = [min_score]
    if stage != "All":
        query += " AND stage = ?"
        params.append(stage)
    if role_keyword:
        query += " AND role LIKE ?"
        params.append(f"%{role_keyword}%")
    query += " ORDER BY score DESC"
    with get_conn() as conn:
        rows = conn.execute(query, params).fetchall()
    return [dict(r) for r in rows]

@mcp.tool()
def add_candidate(name: str, email: str, role: str, experience_years: float, skills: str, stage: str = "Applied", score: int = 50) -> dict:
    with get_conn() as conn:
        cursor = conn.execute(
            "INSERT INTO candidates (name, email, role, experience_years, skills, stage, score) VALUES (?, ?, ?, ?, ?, ?, ?)",
            (name, email, role, experience_years, skills, stage, score)
        )
    return {"status": "created", "candidate_id": cursor.lastrowid}

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

with open(REAL_PROJECT_DIR / "server.py", "w") as f:
    f.write(SERVER_CODE.strip())

MODEL_NAME = "openrouter/free"

async def ask_candidate_tracker(prompt: str, model: str = MODEL_NAME) -> str:
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com",
            "X-OpenRouter-Title": "MCP Candidate Tracker",
        },
    )

    # Removed filesystem server to avoid npx startup connections dropping
    servers = {
        "tracker": StdioServerParameters(
            command=sys.executable,
            args=[str(REAL_PROJECT_DIR / "server.py")],
        )
    }

    sessions: dict[str, ClientSession] = {}
    tool_router: dict[str, tuple[str, str]] = {}
    openai_tools = []

    async with AsyncExitStack() as stack:
        for srv_name, params in servers.items():
            try:
                read, write = await stack.enter_async_context(
                    stdio_client(params, errlog=sys.__stderr__)
                )
                session = await stack.enter_async_context(ClientSession(read, write))
                await session.initialize()
                sessions[srv_name] = session

                tools_resp = await session.list_tools()
                for tool in tools_resp.tools:
                    exposed_name = f"{srv_name}__{tool.name}"
                    tool_router[exposed_name] = (srv_name, tool.name)
                    openai_tools.append(
                        {
                            "type": "function",
                            "function": {
                                "name": exposed_name,
                                "description": tool.description or "",
                                "parameters": tool.inputSchema,
                            },
                        }
                    )
            except Exception as e:
                print(f"Failed to start or connect to server {srv_name}:")
                traceback.print_exc()
                raise e

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an HR recruiting assistant. Call the available MCP tools to inspect "
                    "the SQLite candidate database, then write a clear, well-structured summary report."
                ),
            },
            {"role": "user", "content": prompt},
        ]

        while True:
            resp = await client.chat.completions.create(
                model=model,
                messages=messages,
                tools=openai_tools if openai_tools else None,
            )
            msg = resp.choices[0].message
            messages.append(msg.model_dump(exclude_none=True))

            if not msg.tool_calls:
                return msg.content or ""

            for tc in msg.tool_calls:
                srv_name, orig_tool = tool_router[tc.function.name]
                args = json.loads(tc.function.arguments or "{}")
                print(f"[MCP -> {srv_name}.{orig_tool}({args})]")
                result = await sessions[srv_name].call_tool(orig_tool, arguments=args)
                output_text = "\n".join(
                    c.text for c in result.content if hasattr(c, "text")
                )
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": output_text,
                    }
                )

try:
    report = await ask_candidate_tracker(
        "Give me a summary report of all candidates with score >= 80, grouped by role and stage."
    )
    print(report)
except Exception as e:
    print(f"An exception occurred inside the main execution block: {e}")

[MCP -> tracker.list_candidates({'min_score': 80})]
[MCP -> tracker.list_candidates({})]
[MCP -> tracker.list_candidates({'stage': 'Applied'})]
[MCP -> tracker.list_candidates({'stage': 'Interview'})]
[MCP -> tracker.list_candidates({'stage': 'Offer'})]
[MCP -> tracker.list_candidates({'stage': 'Hired'})]
[MCP -> tracker.list_candidates({'stage': 'Screening'})]
[MCP -> tracker.list_candidates({'min_score': 0})]
**Summary Report – Candidates with Score ≥ 80**

I queried the SQLite candidate database for all records where the score field is 80 or higher, grouping the results by **role** and **stage**.

| Role | Stage | Count |
|------|-------|-------|
| *(none)* | *(none)* | **0** |

**Result:**  
No candidates in the current database meet the score‑of‑80 or above threshold. Consequently, there are no records to group or display.

If you’d like to see candidates with lower scores, add new candidates, or adjust the criteria (e.g., a different minimum score or a specific role/keyword), jus

In [19]:
import json
import os
import sys
from contextlib import AsyncExitStack
from pathlib import Path

import gradio as gr
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import AsyncOpenAI

PROJECT_DIR = Path("/content/mcp_candidate_tracker")
RESUMES_DIR = PROJECT_DIR / "resumes"
RESUMES_DIR.mkdir(parents=True, exist_ok=True)

# Tool-capable free model on OpenRouter (or use "openrouter/free")
MODEL_NAME = "openrouter/free"


class MCPCandidateChatbot:
    def __init__(self):
        self.stack = AsyncExitStack()
        self.sessions: dict[str, ClientSession] = {}
        self.tool_router: dict[str, tuple[str, str]] = {}
        self.openai_tools: list[dict] = []
        self.client = AsyncOpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
            default_headers={
                "HTTP-Referer": "https://colab.research.google.com",
                "X-OpenRouter-Title": "MCP Candidate Tracker Chatbot",
            },
        )
        self.system_prompt = (
            "You are an HR recruiting assistant connected to an MCP tracker "
            "(SQLite candidate database). "
            "Always use the MCP tools to fetch live data before answering, and format "
            "reports clearly with tables or bullet points."
        )

    async def start(self):
        servers = {
            "tracker": StdioServerParameters(
                command=sys.executable,
                args=[str(PROJECT_DIR / "server.py")],
            )
        }

        for srv_name, params in servers.items():
            # We resolve 'UnsupportedOperation: fileno' by specifying errlog=sys.__stderr__
            read, write = await self.stack.enter_async_context(
                stdio_client(params, errlog=sys.__stderr__)
            )
            session = await self.stack.enter_async_context(ClientSession(read, write))
            await session.initialize()
            self.sessions[srv_name] = session

            tools_resp = await session.list_tools()
            for tool in tools_resp.tools:
                exposed_name = f"{srv_name}__{tool.name}"
                self.tool_router[exposed_name] = (srv_name, tool.name)
                self.openai_tools.append(
                    {
                        "type": "function",
                        "function": {
                            "name": exposed_name,
                            "description": tool.description or "",
                            "parameters": tool.inputSchema,
                        },
                    }
                )

    async def chat(self, user_message: str, history: list[list[str]]) -> str:
        # Convert history formatting if it contains lists from older Gradio formats
        messages = [{"role": "system", "content": self.system_prompt}]
        for turn in history:
            if isinstance(turn, (list, tuple)) and len(turn) == 2:
                user_val, assistant_val = turn
                if user_val:
                    messages.append("{\"role\": \"user\", \"content\": \"" + str(user_val) + "\"}")
                if assistant_val:
                    messages.append("{\"role\": \"assistant\", \"content\": \"" + str(assistant_val) + "\"}")
            elif isinstance(turn, dict):
                messages.append({"role": turn["role"], "content": turn["content"]})

        # Clean string formats to dictionary structures if needed
        processed_messages = []
        for m in messages:
            if isinstance(m, str):
                try:
                    processed_messages.append(json.loads(m))
                except Exception:
                    pass
            else:
                processed_messages.append(m)

        processed_messages.append({"role": "user", "content": user_message})
        tool_logs = []

        while True:
            resp = await self.client.chat.completions.create(
                model=MODEL_NAME,
                messages=processed_messages,
                tools=self.openai_tools,
                extra_body={"provider": {"require_parameters": True}},
            )
            msg = resp.choices[0].message
            processed_messages.append(msg.model_dump(exclude_none=True))

            if not msg.tool_calls:
                reply = msg.content or "No response generated."
                if tool_logs:
                    trace = "\n".join(f"- `{t}`" for t in tool_logs)
                    return f"<details><summary>MCP Tools Used ({len(tool_logs)})</summary>\n\n{trace}\n</details>\n\n{reply}"
                return reply

            for tc in msg.tool_calls:
                srv_name, orig_tool = self.tool_router[tc.function.name]
                args = json.loads(tc.function.arguments or "{}")
                tool_logs.append(f"{srv_name}.{orig_tool}({args})")

                result = await self.sessions[srv_name].call_tool(orig_tool, arguments=args)
                output_text = "\n".join(
                    c.text for c in result.content if hasattr(c, "text")
                )
                processed_messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": output_text,
                    }
                )


# Start MCP sessions once and launch Gradio inline in Colab
bot = MCPCandidateChatbot()
await bot.start()

# Removed type="messages" to prevent version compatibility issues
demo = gr.ChatInterface(
    fn=bot.chat,
    title="HR Candidate Tracker (MCP + SQLite)",
    description="Ask in plain English to query SQLite candidates, update stages/scores, or manage hiring metrics.",
    examples=[
        "Give me a complete hiring pipeline summary report grouped by stage and role.",
        "Who are the top candidates with a score above 80?",
        "Move Rohan Kulkarni to Shortlisted and update his score to 76.",
    ],
)

demo.launch(inline=True, share=False)

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-23' coro=<<async_generator_athrow without __name__>()> exception=RuntimeError('Attempted to exit cancel scope in a different task than it was entered in')>
  + Exception Group Traceback (most recent call last):
  |   File "/usr/local/lib/python3.13/dist-packages/anyio/_backends/_asyncio.py", line 871, in __aexit__
  |     raise BaseExceptionGroup(
  |         "unhandled errors in a TaskGroup", self._exceptions
  |     ) from None
  | BaseExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   File "/usr/local/lib/python3.13/dist-packages/mcp/client/stdio/__init__.py", line 189, in stdio_client
    |     yield read_stream, write_stream
    | GeneratorExit
    +------------------------------------

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File 

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>